# xlingual-drift: Colab runner

GPU-only steps run here: serving open-weight models with vLLM, and later the SFT ablation.
Everything else (environment, verifier, stats) is developed locally.

Workflow: clone from GitHub each session, write raw logs to Drive (they survive
disconnects), commit condensed results back to GitHub before the session ends.
Runs are resumable, so after a disconnect just rerun from the top.

In [1]:
# 1. Which GPU did Colab give us? The run config decides whether it is big enough (cell 5).
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

NVIDIA A100-SXM4-80GB, 81920 MiB


In [2]:
# 2. Clone or update the repo. The repo is private, so this uses a GitHub token
#    stored in Colab Secrets as GITHUB_TOKEN (key icon in the left sidebar).
import os
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
if not os.path.exists("/content/xlingual-drift"):
    !git clone -q https://{token}@github.com/khalerishabh/xlingual-drift.git /content/xlingual-drift
else:
    !cd /content/xlingual-drift && git pull -q
%cd /content/xlingual-drift
!git log --oneline -1

/content/xlingual-drift
f0faba1 (HEAD -> main, origin/main, origin/HEAD) Trip pilot 1: pickup scored as 0-60 min after arrival, resample empty replies, amended pilot rule


In [3]:
# 3. Install. The harness needs only light packages, in Colab's own Python.
#    vLLM goes into a separate venv so it never touches Colab's preinstalled
#    torch/torchaudio/RAPIDS; --torch-backend=auto picks the PyTorch build that
#    matches this machine's GPU driver. The vLLM version is pinned by the run config,
#    because it is part of what defines a backbone.
!pip install -q -r requirements.txt
!pip install -q uv
import yaml
CONFIG = "configs/trip_pilot_en.yaml"
GATE_CONFIG = "configs/trip_gate_v1.yaml"
VLLM_VERSION = yaml.safe_load(open(CONFIG))["serving"]["vllm_version"]
if not os.path.exists("/content/vllm-env/bin/vllm"):
    !uv venv -q /content/vllm-env
    !uv pip install -q -p /content/vllm-env/bin/python "vllm=={VLLM_VERSION}" --torch-backend=auto
# ninja: vLLM compiles some kernels at warm-up (e.g. the FP8 fallback on A100) and needs it.
!uv pip install -q -p /content/vllm-env/bin/python ninja
!/content/vllm-env/bin/python -c "import torch, vllm; print('torch', torch.__version__, '| CUDA build', torch.version.cuda, '| GPU visible', torch.cuda.is_available(), '| vllm', vllm.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 113.6 MB/s eta 0:00:00
torch 2.13.0+cu132 | CUDA build 13.2 | GPU visible True | vllm 0.30.0


In [5]:
# 4. Raw logs go to Drive so a disconnect never loses finished episodes.
from google.colab import drive
drive.mount("/content/drive")
RUNS_DIR = "/content/drive/MyDrive/xlingual-drift-runs"
os.makedirs(RUNS_DIR, exist_ok=True)

Mounted at /content/drive


## Start the model server

The model, weights revision, parsers and minimum GPU memory come from the run config's
`serving` block, so the notebook can only serve what the config pins. Gate v2 uses the
official **Qwen3.6-27B-FP8**, which fits a 40GB or 80GB A100. On a smaller GPU the cell stops
and asks you to switch runtime. The first start downloads the weights (roughly 30GB).
Server output goes to `/content/vllm.log`.

In [6]:
# 5. Serve exactly the backbone the run config pins, and record what was served.
#    A backbone is the model plus its exact weights and precision; run_grid
#    refuses to mix episodes from different setups in one log.
import json, subprocess, time, urllib.request, yaml
from huggingface_hub import HfApi

serving = yaml.safe_load(open(CONFIG))["serving"]

gpu_name, vram_mib = [x.strip() for x in subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.split(",")]
vram_gb = int(vram_mib) / 1024
if vram_gb < serving["min_vram_gb"]:
    raise SystemExit(f"{serving['model']} needs >= {serving['min_vram_gb']} GB of GPU memory; this runtime has "
                     f"{gpu_name} ({vram_gb:.0f} GB). Disconnect, choose A100, and reconnect.")

sha = HfApi().model_info(serving["model"], revision=serving["revision"]).sha
vllm_version = subprocess.run(["/content/vllm-env/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                              capture_output=True, text=True).stdout.strip()
if vllm_version != serving["vllm_version"]:
    raise SystemExit(f"vLLM {vllm_version} installed but the config pins {serving['vllm_version']}. "
                     "Delete /content/vllm-env and rerun cell 3.")
SERVING_JSON = "/content/serving.json"
json.dump({"model": serving["model"], "revision": sha, "vllm_version": vllm_version,
           "gpu_name": gpu_name, "vram_gb": round(vram_gb), "max_model_len": serving["max_model_len"]},
          open(SERVING_JSON, "w"), indent=2)
print(open(SERVING_JSON).read())

cmd = ["/content/vllm-env/bin/vllm", "serve", serving["model"],
       "--revision", sha,
       "--port", "8000",
       "--max-model-len", str(serving["max_model_len"]),
       "--gpu-memory-utilization", "0.90",
       "--enable-auto-tool-choice",
       "--tool-call-parser", serving["tool_call_parser"],
       "--reasoning-parser", serving["reasoning_parser"],
       *serving.get("extra_args", [])]
import os
# The venv's bin must be on PATH so vLLM finds build tools (ninja) it installed there.
env = {**os.environ, "PATH": "/content/vllm-env/bin:" + os.environ["PATH"]}
server = subprocess.Popen(cmd, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT, env=env)

for i in range(240):
    try:
        if urllib.request.urlopen("http://localhost:8000/health", timeout=5).status == 200:
            print(f"server up after ~{i * 10}s")
            break
    except Exception:
        pass
    if server.poll() is not None:
        raise RuntimeError("vLLM exited; see the tail of /content/vllm.log below")
    time.sleep(10)
else:
    raise TimeoutError("server not up after 40 min")

{
  "model": "Qwen/Qwen3.6-27B-FP8",
  "revision": "e89b16ebf1988b3d6befa7de50abc2d76f26eb09",
  "vllm_version": "0.30.0",
  "gpu_name": "NVIDIA A100-SXM4-80GB",
  "vram_gb": 80,
  "max_model_len": 65536
}
server up after ~570s


In [ ]:
# If the server failed to start, the reason is here.
!tail -n 30 /content/vllm.log

## Checks before spending GPU time

In [7]:
# 6. Harness sanity: tests and the mock smoke test (no model involved).
!python -m pytest -q
!python -m eval.run_grid --config configs/smoke_test.yaml --out /content/smoke_test.jsonl | tail -2
!python -m eval.run_grid --config configs/smoke_test_trip.yaml --out /content/smoke_test_trip.jsonl | tail -2

........................................................................ [ 43%]
........................................................................ [ 87%]
....................                                                     [100%]
164 passed in 0.55s
Wrote 126 episode(s) to /content/smoke_test.jsonl
Summary: {'n': 126, 'infra_errors': 0, 'success_rate_clean': 0.25, 'recovery_rate': 0.667}
Wrote 36 episode(s) to /content/smoke_test_trip.jsonl
Summary: {'n': 36, 'infra_errors': 0, 'success_rate_clean': 0.143, 'recovery_rate': 1.0}


## Trip domain: English pilot, then the gate

The long-horizon trip task (docs/TRIP_SPEC.md): one request, 11 to 28 required tool calls.
Difficulty is decided on English only. Rule (amended 29 Sep 2026 after the first English pilot,
before any other language ran): if pooled English success over all horizons is below 0.7, stop
here and report back; the task gets simplified before any other language runs.
Otherwise the gate runs in all five languages with nothing changed.
Both steps are safe to rerun after a disconnect: `--resume` skips finished episodes.

In [8]:
# 7. English pilot (second run): 40 episodes. --serving stamps every episode with what was served.
import json
from eval.metrics import latest_scored, rate_table
PILOT_LOG = f"{RUNS_DIR}/trip_pilot_en_v2_qwen36_27b_fp8.jsonl"
!python -m eval.run_grid --config {CONFIG} --serving {SERVING_JSON} --resume --workers 16 --out {PILOT_LOG}

pilot = latest_scored([json.loads(l) for l in open(PILOT_LOG, encoding="utf-8")])
by_h = rate_table(pilot, "success")
print({f"h{h}": f"{rate:.2f} (n={n})" for (_, h), (rate, n) in by_h.items()})
pooled = sum(r["success"] for r in pilot) / len(pilot)
print(f"pooled English success: {pooled:.2f} (n={len(pilot)})")
PILOT_PASSED = pooled >= 0.7
print("decision:", "PASS - design frozen, run the gate" if PILOT_PASSED else "FAIL - stop and report back; do not run the gate")

  [1/40] trip_001 h11 en       qwen36_27b_fp8_think   s4 success=False failed=cab.pickup_time steps=19 recovered=None
  [2/40] trip_001 h11 en       qwen36_27b_fp8_think   s7 success=False failed=cab.pickup_time steps=20 recovered=None
  [3/40] trip_001 h11 en       qwen36_27b_fp8_think   s3 success=True  failed=- steps=17 recovered=None
  [4/40] trip_001 h17 en       qwen36_27b_fp8_think   s5 success=True  failed=- steps=23 recovered=None
  [5/40] trip_001 h11 en       qwen36_27b_fp8_think   s9 success=True  failed=- steps=18 recovered=None
  [6/40] trip_001 h11 en       qwen36_27b_fp8_think   s5 success=True  failed=- steps=24 recovered=None
  [7/40] trip_001 h11 en       qwen36_27b_fp8_think   s8 success=True  failed=- steps=19 recovered=None
  [8/40] trip_001 h11 en       qwen36_27b_fp8_think   s6 success=True  failed=- steps=23 recovered=None
  [9/40] trip_001 h17 en       qwen36_27b_fp8_think   s4 success=True  failed=- steps=27 recovered=None
  [10/40] trip_001 h17 en       qwen

In [9]:
# 7b. The gate in all five languages (60 episodes). Runs only if the pilot passed.
LOG = f"{RUNS_DIR}/trip_gate_v1_qwen36_27b_fp8.jsonl"
if PILOT_PASSED:
    !python -m eval.run_grid --config {GATE_CONFIG} --serving {SERVING_JSON} --resume --workers 16 --out {LOG}
else:
    LOG = PILOT_LOG
    print("pilot failed: showing the pilot in the tables below instead")

  [1/60] trip_001 h11 ta       qwen36_27b_fp8_think   s2 success=True  failed=- steps=19 recovered=None
  [2/60] trip_001 h11 hinglish qwen36_27b_fp8_think   s3 success=True  failed=- steps=20 recovered=None
  [3/60] trip_001 h11 ta       qwen36_27b_fp8_think   s3 success=False failed=return.objective steps=22 recovered=None
  [4/60] trip_001 h11 zh       qwen36_27b_fp8_think   s2 success=True  failed=- steps=16 recovered=None
  [5/60] trip_001 h11 ta       qwen36_27b_fp8_think   s1 success=False failed=return.objective steps=22 recovered=None
  [6/60] trip_001 h11 hinglish qwen36_27b_fp8_think   s1 success=False failed=cab.pickup_time steps=20 recovered=None
  [7/60] trip_001 h11 hinglish qwen36_27b_fp8_think   s2 success=False failed=return.objective,cab.pickup_time steps=22 recovered=None
  [8/60] trip_001 h11 en       qwen36_27b_fp8_think   s3 success=False failed=cab.pickup_time steps=21 recovered=None
  [9/60] trip_001 h11 zh       qwen36_27b_fp8_think   s1 success=False failed=c

In [10]:
# 8. Survival by constraint class, language x horizon (clean episodes).
import collections
records = latest_scored([json.loads(l) for l in open(LOG, encoding="utf-8")])
print(f"{len(records)} scored episodes")
langs = [l for l in ["en", "zh", "hi", "ta", "hinglish"] if any(r["language"] == l for r in records)]
hs = [11, 17, 24, 28]
views = (("success", "full success", False),
         ("all_early_survived", "early: flights", True),
         ("all_mid_survived", "mid: hotel, rooms, cab, tickets", True),
         ("all_carried_survived", "carried: cab pickup time and drop from tool outputs", True),
         ("all_late_survived", "late: per-traveller seats, meals, tickets, rooms (booked only)", True))
for field, label, only_booked in views:
    table = rate_table(records, field, only_booked)
    print(f"\n{label}")
    print("  lang      " + "  ".join(f"h{h:<3}" for h in hs))
    for lang in langs:
        print(f"  {lang:<9} " + "  ".join(f"{table[(lang, h)][0]:.2f}" if (lang, h) in table else " -- " for h in hs))

booked = [r for r in records if r["booked_flight"]]
print("\nbooked:", len(booked), "of", len(records))
print("mean binding swaps (booked):", {l: round(sum(r["binding_swaps"] for r in booked if r["language"] == l) / max(1, sum(r["language"] == l for r in booked)), 2) for l in langs})
print("mean steps:", {l: round(sum(r["num_steps_taken"] for r in records if r["language"] == l) / max(1, sum(r["language"] == l for r in records)), 1) for l in langs})
tok = [r["transcript"]["usage"]["completion_tokens"] for r in records]
print(f"mean completion tokens per episode: {sum(tok) / len(tok):.0f}")

60 scored episodes

full success
  lang      h11   h17   h24   h28 
  en        0.00  1.00  1.00  1.00
  zh        0.67  1.00  1.00  1.00
  hi        1.00  0.67  0.67  0.33
  ta        0.33  0.33  0.00  0.00
  hinglish  0.33  0.33  0.67  1.00

early: flights
  lang      h11   h17   h24   h28 
  en        1.00  1.00  1.00  1.00
  zh        1.00  1.00  1.00  1.00
  hi        1.00  1.00  0.67  0.67
  ta        0.33  0.33  0.33  0.33
  hinglish  0.67  0.33  0.67  1.00

mid: hotel, rooms, cab, tickets
  lang      h11   h17   h24   h28 
  en        1.00  1.00  1.00  1.00
  zh        1.00  1.00  1.00  1.00
  hi        1.00  1.00  1.00  1.00
  ta        1.00  1.00  1.00  1.00
  hinglish  1.00  1.00  1.00  1.00

carried: cab pickup time and drop from tool outputs
  lang      h11   h17   h24   h28 
  en        0.00  1.00  1.00  1.00
  zh        0.67  1.00  1.00  1.00
  hi        1.00  0.67  1.00  1.00
  ta        1.00  0.67  0.67  1.00
  hinglish  0.33  1.00  1.00  1.00

late: per-traveller seat

In [11]:
# 9. Failures: which constraints did not survive.
for r in sorted(records, key=lambda r: (r["language"], r["horizon"], r["seed"])):
    if not r["success"]:
        bad = [k for k, v in (r["constraint_ok"] or {}).items() if not v] or ["no booking"]
        print(f'{r["language"]:<9} h{r["horizon"]:<2} s{r["seed"]} steps={r["num_steps_taken"]:<3} '
              f'limit={r["hit_step_limit"]} swaps={r["binding_swaps"]} failed={",".join(bad)[:150]}')

en        h11 s1 steps=20  limit=False swaps=0 failed=cab.pickup_time
en        h11 s2 steps=21  limit=False swaps=0 failed=cab.pickup_time
en        h11 s3 steps=21  limit=False swaps=0 failed=cab.pickup_time
hi        h17 s3 steps=27  limit=False swaps=0 failed=cab.pickup_time
hi        h24 s3 steps=25  limit=False swaps=0 failed=return.objective
hi        h28 s1 steps=26  limit=False swaps=0 failed=return.objective
hi        h28 s2 steps=25  limit=False swaps=2 failed=seat.outbound.self,seat.return.self
hinglish  h11 s1 steps=20  limit=False swaps=0 failed=cab.pickup_time
hinglish  h11 s2 steps=22  limit=False swaps=0 failed=return.objective,cab.pickup_time
hinglish  h17 s1 steps=22  limit=False swaps=0 failed=return.depart_after,return.objective
hinglish  h17 s3 steps=24  limit=False swaps=0 failed=return.depart_after,return.objective
hinglish  h24 s2 steps=23  limit=False swaps=0 failed=return.objective
ta        h11 s1 steps=22  limit=False swaps=0 failed=return.objective
ta     

## Get the log to your computer

Download the raw logs and put them in `C:\\Rishabh\\xlingual-drift\\runs\\` so it can be analysed
locally. Then disconnect: Runtime -> Disconnect and delete runtime.

In [12]:
# 10. Download the raw logs.
from google.colab import files
files.download(PILOT_LOG)
if LOG != PILOT_LOG:
    files.download(LOG)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>